In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import joblib
import glob

CSV_FILES=glob.glob('/content/sample_data/ppg_data_*.csv')

dfs=[]
for f in CSV_FILES:
    d = pd.read_csv(f)
    dfs.append(d)

df=pd.concat(dfs, ignore_index=True)
df=df[df['status'] == 'NORMAL'].copy()
df=df.dropna(subset=['hr', 'spo2'])

df=df[(df['hr']>40) & (df['hr']<200)]
df=df[(df['spo2']>80) & (df['spo2']<=100)]

hr=df['hr'].values
spo2=df['spo2'].values

print(f"Total NORMAL samples across all files: {len(df)}")


In [ ]:
LOGS=50
HORIZON=150 #~30 s ahead at 0.2 s/step, 30s/0.2s=150steps ahead

X,Y=[],[]
for i in range(LOGS, len(hr) - HORIZON):
    hr_win=hr[i-LOGS:i]
    spo2_win=spo2[i-LOGS:i]

    features=list(hr_win) + list(spo2_win)

    features+=[
        hr_win.mean(), hr_win.std(), hr_win.max() - hr_win.min(),   # HR mean, std, range
        hr_win[-1] - hr_win[0],                                      # HR trend (last - first)
        spo2_win.mean(), spo2_win.std(), spo2_win.max() - spo2_win.min(),
        spo2_win[-1] - spo2_win[0],                                  # SpO2 trend
    ]

    X.append(features)
    Y.append([hr[i + HORIZON], spo2[i + HORIZON]])

X=np.array(X)
Y=np.array(Y)
print(f"Samples for training: {len(X)}, features: {X.shape[1]}")

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, shuffle=False
)

HR R²: 0.955, SpO2 R²: 0.944


In [ ]:
model=MultiOutputRegressor(GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42))

model.fit(X_train, Y_train)

['predict_30s.pkl']

In [ ]:
Y_pred=model.predict(X_test)
hr_r2=r2_score(Y_test[:, 0], Y_pred[:, 0])
spo2_r2=r2_score(Y_test[:, 1], Y_pred[:, 1])
hr_mae=np.mean(np.abs(Y_test[:, 0] - Y_pred[:, 0]))
spo2_mae=np.mean(np.abs(Y_test[:, 1] - Y_pred[:, 1]))

print(f"\nTest set results:")
print(f"  HR   R²={hr_r2:.3f}  MAE={hr_mae:.1f} bpm")
print(f"  SpO2 R²={spo2_r2:.3f}  MAE={spo2_mae:.1f}%")

In [ ]:
joblib.dump({'model': model, 'n_features': X.shape[1], 'logs': LOGS}, 'predict_30s_multi.pkl')
print("\nSaved predict_30s_multi.pkl")